In [1]:
from google import genai
from google.genai import types
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.stores import InMemoryStore
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_google_genai  import ChatGoogleGenerativeAI
from langchain_community.retrievers import BM25Retriever
import requests
from pathlib import Path
import os
from dotenv import load_dotenv
import uuid
import hashlib


load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=30)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)

C:\Users\cmanw\AppData\Local\Temp\ipykernel_34824\3050199681.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader
c:\Users\cmanw\OneDrive\Documents\AI-Projects\Project_1\mcp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class GeminiEmbeddings(Embeddings):
    def __init__(self, api_key, model="gemini-embedding-2"):
        self.client = genai.Client(api_key=api_key)
        self.model = model

    def embed_documents(self, texts):

        embeddings = []
        batch_size = 50
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_contents = [
                    types.Content(
                        parts=[types.Part(text=text)],
                        role="user",
                    )
                    for text in batch
                    ]
                    
            resp = self.client.models.embed_content(
                model=self.model,
                contents=batch_contents
            )

            embeddings.extend(e.values for e in resp.embeddings)

        return embeddings

    def embed_query(self, text):
        resp = self.client.models.embed_content(
            model=self.model,
            contents=[text]
        )
        return resp.embeddings[0].values

gemini_embeddings = GeminiEmbeddings(api_key=api_key)



In [ ]:
docs_list = []
child_docs_list = []
parent_docs_list = []
RAG_NAMESPACE = uuid.UUID('7d5a5286-6df7-4404-b97c-e0938f381c15')
pdf_dir = Path(r"C:\Users\cmanw\OneDrive\Documents\AI-Projects\Project_1\PDF_FOLDER")
for pdf_path in pdf_dir.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    
    with open(pdf_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()

    document_id = file_hash
    
    for doc in docs:
        doc.metadata["pdf_name"] = pdf_path.name
        doc.metadata["document_id"] = document_id
    docs_list.extend(docs)

    
    parent_docs = parent_splitter.split_documents(docs)
    for parent_doc in parent_docs:
        parent_id = uuid.uuid5(RAG_NAMESPACE, parent_doc.page_content)
        parent_doc.metadata["parent_id"] = parent_id
    parent_docs_list.extend(parent_docs)
   
    
    for i, parent_doc in enumerate(parent_docs, start=1):
        child_docs = child_splitter.split_documents([parent_doc])
        for child_doc in child_docs:
            child_id = uuid.uuid5(RAG_NAMESPACE, child_doc.page_content)
            child_doc.metadata["child_id"] = child_id
        child_docs_list.extend(child_docs)

child_texts = [doc.page_content for doc in child_docs_list]
embedded_contents = gemini_embeddings.embed_documents(child_texts)




# vectorstore = Chroma.from_documents(embedding_functions=gemini_embeddings, persist_directory="chroma_db2")

# vectorstore.add_documents(child_docs_list)

In [ ]:
len(child_texts)

In [ ]:
len(embedded_contents)

In [ ]:
len(parent_docs)

In [4]:
import psycopg2 

conn = psycopg2.connect(
    database="vectordb3",
    user="postgres",
    password="newpassword",  # <-- Change this from "password"
    host="127.0.0.1",
    port=5432
)
cursor = conn.cursor()


In [ ]:
for doc in docs_list:
    file_hash = doc.metadata["document_id"]
    pdf_name = doc.metadata["pdf_name"]
    cursor.execute(
        "INSERT INTO documents (file_hash, pdf_name) VALUES (%s, %s) ON CONFLICT (file_hash) DO NOTHING",
        (file_hash, pdf_name)
    )
conn.commit()



In [ ]:
for parent_doc in parent_docs_list:
    parent_docs_id = parent_doc.metadata["parent_id"] 
    hash_id = parent_doc.metadata["document_id"] 
    parent_pages = parent_doc.metadata["page"]  #mark where the page number is coming from
    parent_texts = parent_doc.page_content 
    cursor.execute(
        "INSERT INTO parent_chunks (parent_id, file_hash, page, parent_texts) VALUES (%s,%s,%s,%s) ON CONFLICT (parent_id) DO NOTHING", 
        (str(parent_docs_id), hash_id, parent_pages, parent_texts)
        )
conn.commit()

In [ ]:
for child_doc, embedded_content in zip(child_docs_list, embedded_contents):
    child_id = child_doc.metadata["child_id"]
    child_parent_id = child_doc.metadata["parent_id"]
    child_text = child_doc.page_content
    embedding = embedded_content
    cursor.execute(
        "INSERT INTO  child_chunks (child_id, parent_id, child_text, embeddings) VALUES (%s,%s,%s,%s) ON CONFLICT (child_id) DO NOTHING", 
        (str(child_id), str(child_parent_id), child_text, embedding)
    )
conn.commit()

In [11]:
query = "What is the agenti ai?"
embed_query = gemini_embeddings.embed_query(query)

string_embed = str(embed_query)
print(string_embed)

[-0.0017675913, 0.0168307, -0.007406837, -0.004946684, 0.00018354316, -0.007912928, -0.018742453, -0.004569591, -0.023582147, -0.048452776, -0.014081679, 0.009245749, 0.0035247288, -0.017133126, 0.0090184305, -0.02667454, 0.03664247, 0.0006456997, 0.012541298, 0.0054451395, -0.015097667, -0.0004538982, 0.013116678, 0.013693595, 0.017057898, 0.027609471, -0.009917653, 0.0018967214, -0.02137793, 0.117555335, -0.0032128617, -0.022522531, 0.021239655, -0.004144917, 0.017307391, 0.009328771, -0.018063912, -0.0072707306, 0.012553429, 0.00018792185, 0.00011547643, 0.018295122, 0.010403878, 0.00040016125, -0.024060378, -0.010362295, -0.010145929, -0.00717417, -0.002213467, 0.0076743206, 0.017974945, 0.01726032, -0.0053182426, -0.015553634, 0.012386719, -0.025317814, 0.02343187, -0.0018663825, -0.018568443, 0.016203908, -0.004476, -0.0026731452, 0.012790813, 0.01678655, -0.0015685684, -0.0031594725, -0.0013892015, -0.021746831, 0.035145514, -0.028175248, -0.010163434, 0.0256747, -0.0065855505, 

In [12]:
sql_query = """ 
WITH ranked_child_chunks AS ( 
    SELECT parent_id, (embeddings <=> %s::vector ) AS distance FROM child_chunks 
    ORDER BY embeddings <=> %s::vector ASC
    LIMIT 20
),

deduplicated_parent_ids AS (
    SELECT  parent_id , MIN(distance) as best_distance from ranked_child_chunks
    GROUP BY parent_id
)

SELECT p.parent_id, p.parent_texts, p.page FROM parent_chunks p
JOIN deduplicated_parent_ids d on p.parent_id = d.parent_id
WHERE length(p.parent_texts) > 100 
ORDER BY d.best_distance ASC
LIMIT 5 
"""
cursor.execute(sql_query, (string_embed, string_embed))

In [13]:
results = cursor.fetchall()
results

[('dae85634-34f3-51fe-936d-ddceee258528',
  "What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively correct its action

In [14]:
documents_payload = [
    {"text": result[1]} for result in results
]
documents_payload

[{'text': "What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively correct its actions if needed. In case \u2028\nof fa

In [15]:
from components.openrouter_rerank import OpenRouterRerank 
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [26]:
reranker = OpenRouterRerank(api_key=OPENROUTER_API_KEY)
results = reranker.rerank(query=query, documents=documents_payload, top_n=1)
results

[{'index': 0,
  'score': 0.007547923375900623,
  'source': "What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively cor

In [38]:

context = [result.get('source') for result in results]
print(context)

["What is an \nagent?\nWhile conventional software enables users to streamline and automate workflows, agents are able \nto perform the same workflows on the users’ behalf with a high degree of independence.\nAgents are systems that independently accomplish tasks on your behalf.\nA workflow is a sequence of steps that must be executed to meet the user’s goal, whether that's \nresolving a customer service issue, booking a restaurant reservation, committing a code change, \u2028\nor generating a report.\nApplications that integrate LLMs but don’t use them to control workflow execution—think simple \nchatbots, single-turn LLMs, or sentiment classifiers—are not agents.\nMore concretely, an agent possesses core characteristics that allow it to act reliably and \nconsistently on behalf of a user:\n01 It leverages an LLM to manage workflow execution and make decisions. It recognizes \nwhen a workflow is complete and can proactively correct its actions if needed. In case \u2028\nof failure, it

In [ ]:
str_context = type(str(context))

In [34]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    api_key=api_key,
)

system_prompt = "You are an expert technical writer. Answer using bullet points'"
user_content = f"""Context:
{context}

Question:
{query}"""

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_content)
])

# 4. Extract the final clean text output
print(response.content)

[{'type': 'text', 'text': "An agent is a system that independently accomplishes tasks and performs workflows on a user's behalf with a high degree of independence. \n\nUnlike simple applications that integrate LLMs (such as basic chatbots), an agent is defined by two core characteristics:\n\n1.  **LLM-Driven Execution:** It uses an LLM to manage and decide how a workflow is executed, recognizing when tasks are complete, proactively correcting its actions, and halting if a failure occurs.\n2.  **Tool Access:** It has access to tools that allow it to interact with external systems to gather context or take action, dynamically selecting the appropriate tools based on the current state of the workflow while operating within defined guardrails.", 'extras': {'signature': 'EnEKbwERTTIPW79OJ1GIfgNzNulIZUx8OUiuUcuOQ9lM51Acu4C3f7huYZl4A/k8SRdYFHyzsSBVd1luiG+3f41yJmoAZKzXUD77DVeKZvWl/PIsCENCBjTkxij7rukTsB2ej2HcQwVCJP1wyrM0I8bsew=='}}]


In [ ]:
cursor = conn.cursor()

In [ ]:
corpus = [
    "The quick lantern flickered beneath azure arches.",
    "Curious pigeons whispered secrets on the rooftop.",
    "A lone violin echoed through the empty hall.",
    "Midnight rain drummed softly on the cobblestones.",
    "She folded the paper boat with trembling fingers.",
    "Autumn leaves danced in a lazy spiral.",
    "The old clock chimed thirteen times at dawn.",
    "Velvet shadows stretched across the cobbler's lane.",
    "A distant ocean breeze carried salt and stories.",
]

In [ ]:
import psycopg2 

In [ ]:
conn = psycopg2.connect(
    database= "vectordb2",
    user= "postgres",
    password= "newpassword",
    host="127.0.0.4",
    port=5432
)
cursor = conn.cursor()
# for text, emb in zip(corpus, embedding):
#     cursor.execute("INSERT INTO items (text, embedding) VALUES (%s, %s)", (text, emb.tolist()))

# conn.commit()
# result = cursor.fetchall()
# result

In [ ]:
cursor = conn.cursor()

In [ ]:
for text, emb in zip(corpus, embedding):
    cursor.execute("INSERT INTO items (text, embedding) VALUES (%s, %s)", (text, emb.tolist()))

conn.commit()

In [ ]:
query_emb = embedding[0].tolist()
cursor.execute(f"SELECT * FROM items ORDER BY embedding <-> '{query_emb}' LIMIT 2")

In [ ]:
result = cursor.fetchall()
result

In [ ]:
result